# Low-Rank Adaptation(LoRA)

**LoRA**是一种微调技术，直接翻译就叫做**低秩适配**，这种方法能够大大减少微调大模型的参数量，从而降低微调的成本，我们看到很多的模型(比如Stable Diffusion)可能都使用了这个技术进行微调，通过LoRA我们也能用预训练模型训练属于自己的个性化模型，下面我们就来介绍一下LoRA

<div align="center">
    <img src="./imgs/LoRA_architecture.png" alt="LoRA_architecture" style="width: 350px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/646831196)

[LoRA 论文](https://arxiv.org/pdf/2106.09685)

[LoRA Pytorch 实现](https://github.com/microsoft/LoRA/blob/main/loralib/layers.py)

## 前置知识

### 正交矩阵

对于$n$阶方阵$A$，若满足$A^TA = AA^T = I$，$I$为单位矩阵，那么我们称$A$为(标准)正交矩阵，它满足:

- $A^T = A^{-1}$

- 列向量组/行向量组都是标准正交向量组，也就是对于任意两个列向量/行向量，它们之间正交且模长都是1

- 行列式$\text{det}(A)$为1或-1



### 矩阵的秩

矩阵$A \in \mathbb{R}^{m \times n}$的秩$\text{rank}(A)$，为这个矩阵的行向量或列向量中最大的线性无关向量的数量，并且满足:

- 列秩与行秩相同，也就是矩阵最大的线性无关的行数与最大的线性无关的列数相同

- $\text{rank}(A) \le \text{min}(m, n)$，显而易见，因为最多的线性无关的向量数肯定不超过行向量数或列向量数，而行秩和列秩相同，那么肯定不能超过二者中小的那一个

- $\text{rank}(A^TA) = \text{rank}(A)$

也就是说，除了线性无关的一组行/列向量，其他行/列向量的**信息冗余**，因为它们可以通过这个线性无关向量组进行线性组合表示出来，秩就可以理解成矩阵的实际信息量

### 特征值和特征向量

对于方阵$A \in \mathbb{R}^{n \times n}$，若存在**非零**向量$x$和标量$\lambda$满足$Ax = \lambda x$，那么称$\lambda$为$A$的特征值，$x$为对应于$\lambda$的特征向量，特征值通过方程$\text{det}(A - \lambda I) = 0$得到，对应的特征向量通过求解$(A - \lambda I)x = 0$得到，它们有如下性质:

- 不同特征值对应的特征向量线性无关

- $\sum \lambda_i = \text{tr}(A)$，$\prod \lambda_i = \text{det}(A)$，其中$\text{tr}(A)$为矩阵$A$的对角线之和，称之为迹

- 非零特征值的数量小于等于$\text{rank}(A)$

- 对于诸如$A^TA$这样的半正定矩阵，非零特征值的个数就是矩阵的秩


### 奇异值分解

对于任意矩阵$A \in \mathbb{R}^{m \times n}$，可以分解为如下形式:

$$
A = U \Sigma V^T
$$

其中:

- $U$: 左奇异矩阵，为一个$m \times m$的方阵，它是一个正交矩阵

- $\Sigma$: 奇异值对角矩阵，只有主对角线有元素(奇异值)，其余值都为零，满足$\sigma_1 \ge \sigma_2 \ge \cdots \ge \sigma_k > 0$，形状为$m \times n$，且由之前的知识可知，非零奇异值数量为$\text{rank}(A)$

- $V$: 右奇异矩阵，为一个$n \times n$的方阵，同样为一个正交矩阵

分解方式如下:

1. 构造$A^TA$，计算它的所有特征值$\lambda_i$和对应单位特征向量$v_i$(从大到小排序特征值和对应特征向量)，计算出奇异值$\sigma_i = \sqrt{\lambda_i}$，把所有单位特征向量按列拼接得到右奇异矩阵$V = [v_1, \cdots, v_n]$，把奇异值$\sigma_i$放在$\Sigma$的第(i, i)个位置，得到奇异值对角矩阵$\Sigma \in \mathbb{R^{m \times n}} = 
\begin{pmatrix}
\sigma_1 & & & & \\
& \sigma_2 & & & \\
& & \ddots & & \\
& & & \sigma_r & \\
& & & & \boldsymbol{0}
\end{pmatrix}
$

2. 构造左奇异矩阵$U$的前$r$列向量，通过公式$u_i = \frac{1}{\sigma_i} Av_i$得到

3. 若$r < m$，则通过补充标准正交基的方法，补全剩余列$b_1, \cdots, b_{m-r}$，得到最终的左奇异矩阵$U = [u_1, \cdots, u_r, b_1, \cdots, b_{m-r}]$


为了方便假设$A \in \mathbb{R}^{r \times r}$，我们记$U = [u_1, \cdots, u_r]$， $V = [v_1, \cdots, v_r]$，$\Sigma$为$r \times r$的方阵，那么把$A = U \Sigma V^T$展开可以得到:

$$
\begin{align*}
A &= U \Sigma V^T 

\\&= \begin{bmatrix} u_1 & \cdots & u_r \end{bmatrix}
\begin{bmatrix}
\sigma_1 & & 0 \\
& \ddots & \\
0 & & \sigma_r
\end{bmatrix}
\begin{bmatrix} v_1^T \\ \vdots \\ v_r^T \end{bmatrix} 

\\&= \begin{bmatrix} u_1 \sigma_1 & \cdots & u_r \sigma_r \end{bmatrix} \begin{bmatrix} v_1^T \\ \vdots \\ v_r^T \end{bmatrix} 

\\&= \sigma_1 u_1 v_1^T + \sigma_2 u_2 v_2^T + \cdots + \sigma_r u_r v_r^T 

\\&= \sum_{i=1}^{r} \sigma_i \, u_i v_i^T
\end{align*}
$$

我们可以发现，如果有的奇异值$\sigma_i$很小，那么忽略它以及对应的两个向量$u_i$和$v_i$对于最后的结果影响很小，那么我们可以**只保留前$k$个**$\sigma_i$和对应的$u_i$和$v_i$，你可以把$\sigma_i$理解成第$i$个方向的影响力，我们只保留影响力最大的$k$个方向，就有:

$$
\begin{align*}
A &\approx U^* \Sigma^* V^{*T}

\\&= \begin{bmatrix} u_1 & \cdots & u_k \end{bmatrix}
\begin{bmatrix}
\sigma_1 & & 0 \\
& \ddots & \\
0 & & \sigma_k
\end{bmatrix}
\begin{bmatrix} v_1^T \\ \vdots \\ v_k^T \end{bmatrix} 
\end{align*}
$$

也就是说，我们能够通过奇异值分解并保留$k$个分量的方式，来对原始的$A$矩阵做**信息压缩**，即**只保留最重要的信息**

## LoRA原理

### LoRA的由来

假设我们模型的预训练参数为$W \in \mathbb{R}^{d^2}$，微调时的更新量为$\Delta W$，那么我们微调后模型的向前传播过程记作$h = Wx + \Delta Wx$，我们知道，预训练模型的参数量是非常大的，如果进行全量微调，需要的计算资源会非常大，因此我们要想办法**降低微调的参数量**

而我们在前置知识部分已经知道，我们可以对$\Delta W$进行**奇异值分解(SVD)**，然后只保留最大的少量奇异值以及它们对应的$u$和$v$向量从而在减少矩阵规模的同时近似原始的$\Delta W$，这样就能大大降低我们需要微调的参数量了

那么，我们记$B = U \Sigma$，$A = V^T$，那么对$\Delta W$进行SVD就有$\Delta W = BA$，然后我们只保留前$r$个大的奇异值，那么就有$B = U^* \Sigma^* = \begin{bmatrix} u_1 \sigma_1 & \cdots & u_r \sigma_r \end{bmatrix}$，$A = V^{*T} = \begin{bmatrix} v_1 & \cdots & v_r \end{bmatrix}^T$，把$B$和$A$两个矩阵当作可训练矩阵(也就是学习对$\Delta W$进行SVD再取前$r$个奇异值的过程)，训练的参数量就从$d^2$变为$2 \times d \times r$，$r$通常取一个很小的值，远小于$d$，那么微调所需更新的参数量就大大降低了

那么Low Rank，也就是低秩这个概念是怎么来的？我们的$B$和$A$两个矩阵都包含$r$个列向量，而这些列向量都是从SVD中的$U$和$V$两个正交矩阵(准确来说，$B$是从$U \Sigma$中取的，但只是改变了$U$中列向量的长度而已，列向量之间依旧正交)取r个正交的列向量得到的，它们肯定线性无关，那么两个矩阵的秩就是$r$，而我们取的$r$由通常很小，所以两个矩阵都是**低秩矩阵**，也就是该方法中低秩一词的由来

### 实际实现

在**实际实现**时，微调后参数实际上是$W' = W + \frac{\alpha}{r}BA$，其中$\alpha$和$r$是我们设定的超参数，微调开始时把$B$初始化为0，$A$使用随机高斯初始化，相当于微调从**什么都不改变**开始进行

使用前面的缩放因子$\frac{\alpha}{r}$，论文中说$\alpha$近似于学习率，除以$r$可以理解为，当保留的秩比较小时，信息丢失比较多，需要放大$BA$来弥补这个损失；而当保留的秩比较多时，信息丢失少但可能引入噪声，需要对$BA$进行减小来削弱噪声的影响，所以作者说这样能**在调整$r$时减少超参数的调整**，也就是不需要为新设定的$r$再找一个合适的学习率，因为这个缩放因子使得微调更新量对于不同的$r$大致接近

也就是说，在第一次尝试时把$r$和$\alpha$设定为相同的值，之后就可以固定$\alpha$，只改变$r$而不调整别的超参数(比如学习率)了

## LoRA代码简单实现

下面我们按照之前介绍的原理，实现一个**简单**的LoRA代码，如果想知道具体如何微调大模型的，可以看我在开头贴出的源码链接

In [9]:
import torch
import torch.nn as nn


# 以线性层为例实现LoRA
class LoRALinear(nn.Module):
    def __init__(self, in_features:int, out_features:int, rank:int, alpha:float, 
                 dropout:float, freeze_origin:bool = True):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha/rank

        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)
        self._init_A_B_weights()

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        if freeze_origin:
            for para in self.linear.parameters():
                para.requires_grad = False

    def _init_A_B_weights(self):
        nn.init.normal_(self.lora_A.weight, std=1e-5)
        nn.init.zeros_(self.lora_B.weight)
        
    def forward(self, x:torch.Tensor):
        origin_out = self.linear(x)
        lora_out =  self.scaling*self.lora_B(self.dropout(self.lora_A(x)))

        return origin_out + lora_out
    
    def merge_lora(self):
        with torch.no_grad():
            # [out_features, rank]×[rank, in_features] = [out_features, in_features]
            lora_weight = self.lora_B.weight @ self.lora_A.weight
            lora_weight = lora_weight * self.scaling
            self.linear.weight.data += lora_weight

        del self.lora_A, self.lora_B
        return self.linear
    
    def print_trainable_params(self):
        for name, param in self.named_parameters():
            if param.requires_grad:
                print(f"param name:{name:<10} | shape:{param.shape}")
        total = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"total trainable counts: {total}")


In [10]:
lora_linear_layer = LoRALinear(in_features=512, out_features=512, rank=8, alpha=8, dropout=0.1)
x = torch.randn(2, 10, 512)
y = lora_linear_layer(x)
print('in lora linear layer:')
print(f'input x size: {x.size()}')
print(f'output y size: {y.size()}')
print('trainable param:')
lora_linear_layer.print_trainable_params()


merged_lora_linear_layer = lora_linear_layer.merge_lora()
x = torch.randn(2, 10, 512)
y = merged_lora_linear_layer(x)
print('\nin merged lora linear layer:')
print(f'input x size: {x.size()}')
print(f'output y size: {y.size()}')


in lora linear layer:
input x size: torch.Size([2, 10, 512])
output y size: torch.Size([2, 10, 512])
trainable param:
param name:lora_A.weight | shape:torch.Size([8, 512])
param name:lora_B.weight | shape:torch.Size([512, 8])
total trainable counts: 8192

in merged lora linear layer:
input x size: torch.Size([2, 10, 512])
output y size: torch.Size([2, 10, 512])
